Load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [14]:
import pandas as pd

df = pd.read_csv('../data/raw/ethiopia_fi_unified_data.csv')
df.head()

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


In [26]:
# Examine overall structure
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

print('\nColumn dtypes:')
print(df.dtypes)

# Basic summary stats for numeric columns
summary = df.describe(include='all').T
summary

Shape: (43, 34)

Columns:
['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']

Column dtypes:
record_id                  str
record_type                str
category                   str
pillar                     str
indicator                  str
indicator_code             str
indicator_direction        str
value_numeric          float64
value_text                 str
value_type                 str
unit                       str
observation_date           str
period_start               str
period_end                

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
record_id,43,43,REC_0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
record_type,43,3,observation,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,10,7,product_launch,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pillar,33,4,ACCESS,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
indicator,43,29,Account Ownership Rate,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
indicator_code,43,29,ACC_OWNERSHIP,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
indicator_direction,33,3,higher_better,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
value_numeric,33.0,NaN,NaN,NaN,94372576990.694839,423106092501.167664,1.08,24.0,61.4,15000000.0,2380000000000.0
value_text,10,3,Launched,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
value_type,43,6,percentage,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# Explore record types and basic distributions

# How many of each record_type do we have?
record_type_counts = df['record_type'].value_counts()
print(record_type_counts)

# Distinct indicators by record_type
indicators_by_type = df.groupby('record_type')['indicator'].nunique()
print('\nDistinct indicators by record_type:')
print(indicators_by_type)

# Example: look at a few rows of each type
sample_by_type = df.groupby('record_type').head(3)
sample_by_type[['record_id', 'record_type', 'category', 'pillar', 'indicator', 'value_numeric', 'unit', 'observation_date']].head(15)

record_type
observation    30
event          10
target          3
Name: count, dtype: int64

Distinct indicators by record_type:
record_type
event          10
observation    19
target          3
Name: indicator, dtype: int64


,record_id,record_type,category,pillar,indicator,value_numeric,unit,observation_date
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,22.0,%,2014-12-31
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,35.0,%,2017-12-31
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,46.0,%,2021-12-31
30,REC_0031,target,NaN,ACCESS,Account Ownership Rate,70.0,%,2025-12-31
31,REC_0032,target,NaN,ACCESS,Fayda Digital ID Enrollment,90000000.0,people,2028-12-31
32,REC_0033,target,NaN,GENDER,Female Mobile Money Account Share,50.0,%,2030-12-31
33,EVT_0001,event,product_launch,NaN,Telebirr Launch,NaN,NaN,2021-05-17
34,EVT_0002,event,market_entry,NaN,Safaricom Ethiopia Commercial Launch,NaN,NaN,2022-08-01
35,EVT_0003,event,product_launch,NaN,M-Pesa Ethiopia Launch,NaN,NaN,2023-08-01


## Record types in the unified dataset

The `record_type` column classifies each row into three conceptual layers:

- **observation**: Actual measured values from **surveys, reports, or operators** (e.g. Global Findex, Ethio Telecom, EthSwitch). These are realized outcomes in the market.
- **event**: **Policies, product launches, market entries, infrastructure rollouts, milestones, and pricing changes** that may influence the observations over time.
- **target**: **Official policy goals or strategic targets** (e.g. NFIS-II, NBE, Fayda) that define where the system is *intended* to go.

All records share the same column structure, which allows us to join, filter, and model relationships between **observations**, **events**, and **targets** within a single tidy table.